# NER de domínio nos abstracts do arXiv

Pega os `papers.json` que a DAG `arxiv_dag` grava em `data/processed/` e monta um pipeline do spaCy que
reconhece entidades da área (**MODEL**, **DATASET** e **METRIC**), usando o que eu vi nas aulas:
`EntityRuler`, `Matcher`, componente customizado e regex com `filter_spans`.
No final compara com o `en_core_web_sm` puro.

In [1]:
# instala as libs (o "!" manda o comando pro shell, fora do Python)
!pip install spacy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 3.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.8/260.8 kB 5.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.1/134.1 kB 2.5 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.0/869.0 kB 3.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 5.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 2.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 5.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 6.6 MB/s eta 0:00:00


In [2]:
# modelo pequeno de inglês: os abstracts do arXiv estão em inglês
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 4.6 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [3]:
import glob
import json
import re
from collections import Counter

import pandas as pd
import spacy
from spacy import displacy
from spacy.language import Language
from spacy.matcher import Matcher
from spacy.util import filter_spans

## Carregando os papers que a DAG baixou

In [4]:
# a DAG grava um papers.json por dia em data/processed/<data>/papers.json
caminhos = sorted(glob.glob("data/processed/*/papers.json"))
print(len(caminhos), "dias encontrados")
assert len(caminhos) > 0, "nenhum papers.json, roda a DAG primeiro (ou confere se o notebook tá na raiz do projeto)"

# junta tudo num dict pelo id, assim o mesmo paper não entra duas vezes
papers = {}
for caminho in caminhos:
    with open(caminho, "r", encoding="utf-8") as f:
        for p in json.load(f):
            papers[p["id"]] = p

df = pd.DataFrame(papers.values())
print(len(df), "papers únicos")
df[["id", "titulo", "primary_category", "published"]].head()

2 dias encontrados
340 papers únicos


,id,titulo,primary_category,published
0,2610.04824v1,Agent Behavior as Code: Efficient and Robust L...,cs.AI,2026-10-04
1,2610.06977v1,Visual-Invariance-Augmented Feature Optimal Al...,cs.CV,2026-10-04
2,2610.04825v1,DICE: Decoupling Capability from Intervention ...,cs.AI,2026-10-04
3,2610.04829v1,Viva La Vida: Verification and Accumulation Fa...,cs.CL,2026-10-04
4,2610.04830v1,Cluster Validation Indices as Self-Supervised ...,cs.CL,2026-10-04


In [5]:
# quantos abstracts usar (500 já dá uma estatística boa e roda rápido)
N = 500
ids = df.id.tolist()[:N]
titulos = df.titulo.tolist()[:N]
textos = df.resumo.tolist()[:N]

print(len(textos), "abstracts")
print(textos[0])

340 abstracts
AI agents based on foundation models (FMs) have demonstrated strong capabilities to perform complex open-ended tasks. However, they face some common challenges in practice: (a) agent behavior can deviate drastically even for semantically similar tasks, leading to catastrophically propagated errors; (b) high cost and latency due to FM calls, repeated in full whenever a task recurs with different inputs; (c) FMs' limited context and instruction following capability confine how well agents manage the ever-growing execution context and follow complex plans. We introduce $\textbf{A}$gent $\textbf{B}$ehavior as $\textbf{C}$ode $\textbf{Agent}$ (ABCAgent), which uses a symbolic program (e.g., Python code with potential neural functions) to fully specify the agent's behavior at runtime, with a powerful FM agent editing that program for flexibility. Behavior is thus specified without premature variable binding, and its execution is deterministic. We evaluate ABCAgent on six agent 

## Baseline: o modelo pré-treinado puro

In [6]:
nlp_base = spacy.load("en_core_web_sm")

# nlp.pipe processa uma lista de textos de uma vez, é bem mais rápido que chamar nlp(texto) dentro de um for
docs_base = list(nlp_base.pipe(textos))

# conta quantas entidades de cada tipo o modelo puro achou
contagem_base = Counter(ent.label_ for doc in docs_base for ent in doc.ents)
print(contagem_base.most_common())

[('CARDINAL', 1376), ('ORG', 1214), ('PERCENT', 320), ('PERSON', 236), ('GPE', 131), ('ORDINAL', 113), ('MONEY', 84), ('NORP', 78), ('PRODUCT', 74), ('DATE', 54), ('LANGUAGE', 26), ('QUANTITY', 20), ('WORK_OF_ART', 18), ('TIME', 16), ('LOC', 14), ('FAC', 7), ('EVENT', 6), ('LAW', 3)]


In [7]:
# o que ele acha num abstract só (repara que não existe MODEL, DATASET nem METRIC no vocabulário dele)
doc = docs_base[0]
for ent in doc.ents:
    print(ent.text, "->", ent.label_)

AI -> ORG
$\textbf{C}$ode $ -> MONEY
ABCAgent -> ORG
six -> CARDINAL
two -> CARDINAL
ABCAgent -> ORG
GAIA -> ORG
98.3% -> PERCENT
97.3% -> PERCENT
GSM-Symbolic -> ORG
p -> MONEY
0.001$ -> MONEY
71.9% -> PERCENT
47.4% -> PERCENT
$\mathrm{Pass}^4$ -> MONEY
$p = -> MONEY
0.0001$ -> MONEY
WorkArena -> ORG
ABCAgent -> ORG
ABCAgent -> ORG
92.6% -> PERCENT
GSM-Symbolic -> ORG
20.1% -> PERCENT
5.2\times$ -> MONEY
7.0\times$ -> MONEY
GSM-Symbolic -> ORG
19% -> PERCENT
9.5\times$ -> MONEY


## EntityRuler com as listas de termos da área

Mesma ideia da live 11 (listas -> padrões), só que aqui as listas são escritas na mão.

In [8]:
# rótulos que a gente vai criar
ROTULOS = ["MODEL", "DATASET", "METRIC"]

modelos = [
    "BERT", "RoBERTa", "DistilBERT", "GPT-2", "GPT-3", "GPT-3.5", "GPT-4", "GPT-4o",
    "T5", "BART", "LLaMA", "Llama", "Mistral", "Mixtral", "Gemma", "Qwen", "DeepSeek",
    "Claude", "Gemini", "PaLM", "CLIP", "ViT", "ResNet", "VGG", "EfficientNet", "U-Net",
    "YOLO", "Stable Diffusion", "LSTM", "GRU", "CNN", "RNN", "GAN", "VAE", "Mamba",
    "Transformer", "Transformers", "XGBoost",
]

datasets = [
    "ImageNet", "CIFAR-10", "CIFAR-100", "MNIST", "Fashion-MNIST", "COCO", "MS COCO",
    "SQuAD", "GLUE", "SuperGLUE", "MMLU", "GSM8K", "HumanEval", "WikiText", "Common Crawl",
    "LibriSpeech", "MIMIC-III", "MIMIC-IV", "FER2013", "CelebA", "KITTI", "Pascal VOC",
]

# "precision" e "recall" ficaram de fora de propósito: aparecem demais em outros sentidos (mixed precision, por exemplo)
# palavra comum aparece com maiúscula no começo da frase, por isso tem as duas versões
metricas = [
    "accuracy", "Accuracy", "top-1 accuracy", "top-5 accuracy", "F1", "F1-score", "F1 score",
    "AUC", "AUROC", "BLEU", "ROUGE", "perplexity", "Perplexity", "mAP", "RMSE", "MAE", "MSE",
]

In [9]:
nlp = spacy.load("en_core_web_sm")

# before="ner": o ruler roda ANTES do ner e as regras ganham prioridade (live 5)
ruler = nlp.add_pipe("entity_ruler", before="ner")

patterns = []
for m in modelos:
    patterns.append({"label": "MODEL", "pattern": m})
for d in datasets:
    patterns.append({"label": "DATASET", "pattern": d})
for m in metricas:
    patterns.append({"label": "METRIC", "pattern": m})

ruler.add_patterns(patterns)
print(len(patterns), "padrões")

77 padrões


In [10]:
# confere a ordem: entity_ruler tem que aparecer antes do ner
# (na live a gente usou o nlp.analyze_pipes(), que mostra mais detalhe mas despeja um dict enorme)
print(nlp.pipe_names)

['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'entity_ruler', 'ner']


## Componente customizado: regex pra versões de modelo

Lista fixa nunca vai ter `GPT-5`, `Llama 4`, `Qwen2.5`... Então um componente com regex + `char_span`
(live 10) pega a família do modelo seguida do número da versão.

In [11]:
# família do modelo + (hífen ou espaço) + versão. Ex: GPT-5, Llama 4, Qwen2.5, DeepSeek-V3, Llama-3-8B
padrao_versao = r"\b(?:GPT|Llama|LLaMA|Qwen|Gemma|Mistral|Gemini|Claude|DeepSeek|Phi|ResNet)[- ]?[VR]?\d+(?:\.\d+)?[a-z]?(?:-[A-Za-z0-9]+)*\b"


@Language.component("modelo_regex")
def modelo_regex(doc):
    novos = []
    for match in re.finditer(padrao_versao, doc.text):
        # o regex trabalha em posição de caractere, o doc.ents em posição de token: o char_span faz a conversão
        # e já dá o rótulo. Devolve None se o trecho não bater com o limite de um token
        span = doc.char_span(match.start(), match.end(), label="MODEL")
        if span is not None:
            novos.append(span)

    # filter_spans fica só com o span mais longo quando dois se sobrepõem. Sem ele, doc.ents dá erro
    # Os novos vão primeiro na lista pra ganhar em caso de empate com o que o ner marcou
    doc.ents = filter_spans(novos + list(doc.ents))
    return doc

In [12]:
# depois do ner, assim ele corrige o que o modelo marcou errado (ex: "GPT-5" como ORG)
nlp.add_pipe("modelo_regex", after="ner")
print(nlp.pipe_names)

['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'entity_ruler', 'ner', 'modelo_regex']


In [13]:
# teste rápido numa frase inventada: versões que não estão na lista
teste = nlp("We compare GPT-5, Llama 4 and Qwen2.5 with DeepSeek-V3 on ImageNet, reaching 91.5% accuracy.")
for ent in teste.ents:
    print(ent.text, "->", ent.label_)

GPT-5 -> MODEL
Llama 4 -> MODEL
Qwen2.5 -> MODEL
DeepSeek-V3 -> MODEL
ImageNet -> DATASET
91.5% -> PERCENT
accuracy -> METRIC


## Rodando nos abstracts e comparando com o modelo puro

In [14]:
docs = list(nlp.pipe(textos))

contagem = Counter(ent.label_ for doc in docs for ent in doc.ents)

# lado a lado: modelo puro x nosso pipeline (0 = não achou nenhuma desse tipo)
tabela = pd.DataFrame({"modelo_puro": contagem_base, "nosso_pipeline": contagem}).fillna(0).astype(int)
tabela.sort_values("nosso_pipeline", ascending=False)

,modelo_puro,nosso_pipeline
CARDINAL,1376,1374
ORG,1214,1180
PERCENT,320,320
PERSON,236,223
METRIC,0,147
MODEL,0,116
ORDINAL,113,113
GPE,131,103
MONEY,84,83
NORP,78,76


In [15]:
# em quantos abstracts apareceu pelo menos 1 entidade de cada tipo
for rotulo in ROTULOS:
    com = sum(1 for doc in docs if any(ent.label_ == rotulo for ent in doc.ents))
    print(rotulo, "->", com, "de", len(docs), "abstracts (%.0f%%)" % (100 * com / len(docs)))

MODEL -> 72 de 340 abstracts (21%)
DATASET -> 6 de 340 abstracts (2%)
METRIC -> 95 de 340 abstracts (28%)


In [16]:
# as 10 entidades mais frequentes de cada tipo
for rotulo in ROTULOS:
    topo = Counter(ent.text for doc in docs for ent in doc.ents if ent.label_ == rotulo)
    print(rotulo, topo.most_common(10))

MODEL [('Transformer', 9), ('Claude', 6), ('Qwen', 5), ('Llama', 5), ('Transformers', 4), ('Qwen3-4B', 4), ('VAE', 4), ('Qwen3-8B', 4), ('BERT', 3), ('GPT-5.5', 3)]
DATASET [('GSM8K', 4), ('MMLU', 2), ('CIFAR-100', 1), ('GLUE', 1), ('HumanEval', 1)]
METRIC [('accuracy', 97), ('F1', 26), ('AUROC', 9), ('AUC', 6), ('perplexity', 4), ('top-1 accuracy', 2), ('MSE', 1), ('F1 score', 1), ('Accuracy', 1)]


In [17]:
# e o que o modelo puro dizia sobre os termos das nossas listas? (geralmente rótulo errado: ORG, PERSON, PRODUCT...)
termos = set(modelos + datasets + metricas)
rotulos_base = Counter((ent.text, ent.label_) for doc in docs_base for ent in doc.ents if ent.text in termos)
rotulos_base.most_common(15)

[(('F1', 'GPE'), 17),
 (('Transformer', 'ORG'), 5),
 (('Qwen', 'PERSON'), 4),
 (('BERT', 'ORG'), 3),
 (('VAE', 'ORG'), 3),
 (('GSM8K', 'ORG'), 3),
 (('Mamba', 'ORG'), 2),
 (('Llama', 'PRODUCT'), 2),
 (('F1', 'ORG'), 2),
 (('U-Net', 'ORG'), 2),
 (('Claude', 'ORG'), 1),
 (('Mistral', 'ORG'), 1),
 (('Gemma', 'PERSON'), 1),
 (('Llama', 'DATE'), 1),
 (('Gemma', 'ORG'), 1)]

## Matcher: resultados reportados e comparações entre modelos

Agora que `MODEL` e `METRIC` existem como entidades, o `Matcher` pode usar `ENT_TYPE` nos padrões (live 7).

In [18]:
matcher = Matcher(nlp.vocab)

# número (+ % opcional) + métrica. Ex: "92.3% accuracy", "0.91 F1"
resultado1 = [{"LIKE_NUM": True}, {"ORTH": "%", "OP": "?"}, {"ENT_TYPE": "METRIC", "OP": "+"}]

# métrica + "of" ou "=" + número. Ex: "accuracy of 92.3%", "F1 = 0.91"
resultado2 = [{"ENT_TYPE": "METRIC", "OP": "+"}, {"LOWER": {"IN": ["of", "="]}}, {"LIKE_NUM": True}, {"ORTH": "%", "OP": "?"}]

matcher.add("RESULTADO", [resultado1, resultado2], greedy="LONGEST")

# MODEL + verbo de superar (pelo lema) + MODEL. Ex: "Mamba outperforms Transformers"
supera = [
    {"ENT_TYPE": "MODEL", "OP": "+"},
    {"LEMMA": {"IN": ["outperform", "surpass", "beat"]}},
    {"ENT_TYPE": "MODEL", "OP": "+"},
]
matcher.add("SUPERA", [supera], greedy="LONGEST")

In [19]:
achados = []
for doc in docs:
    matches = matcher(doc)
    matches.sort(key=lambda x: x[1])  # ordem de aparição no texto (igual na live 7)
    for match_id, inicio, fim in matches:
        achados.append((nlp.vocab.strings[match_id], doc[inicio:fim].text, doc[inicio:fim].sent.text))

df_achados = pd.DataFrame(achados, columns=["tipo", "trecho", "frase"])
print(df_achados.tipo.value_counts())

pd.set_option("display.max_colwidth", 130)
df_achados.head(15)

tipo
RESULTADO    17
Name: count, dtype: int64


,tipo,trecho,frase
0,RESULTADO,40.48% accuracy,"On Video-BrowseComp, the VideoResearchAgent trained using Qwen3.5-4B achieves 40.48% accuracy, comparable to Gemini-3-Flash-Pr..."
1,RESULTADO,0.846 AUROC,These traces make Verification Trap predictable before hidden execution: a lightweight gold-free predictor using verifier-visi...
2,RESULTADO,2.8 F1,Key findings reveal that attention pooling consistently outperforms alternative strategies under noisy conditions (~+2.6-2.8 F...
3,RESULTADO,~5 F1,Key findings reveal that attention pooling consistently outperforms alternative strategies under noisy conditions (~+2.6-2.8 F...
4,RESULTADO,F1 = 0.9194,"SALUS achieves F1 = 0.9194, significantly outperforming the state-of-the-art baselines."
5,RESULTADO,3.2% accuracy,"On six multimodal datasets, DREAM improves the accuracy-token trade-off over multi-agent debate baselines by 1.5-3.2% accuracy..."
6,RESULTADO,AUC of 0.642,"On Alliance/CALGB 89803, the single model achieves an AUC of 0.642, while the post hoc ensemble reaches a pooled out-of-fold A..."
7,RESULTADO,AUC of 0.70,"On Alliance/CALGB 89803, the single model achieves an AUC of 0.642, while the post hoc ensemble reaches a pooled out-of-fold A..."
8,RESULTADO,F1 of five,"Across 2,586 human-labeled criteria from 100 personal-assistant episodes, Sauron achieves the highest F1 of five verifiers on ..."
9,RESULTADO,0.96 F1,"With GPT-5.6 Sol, it reaches 0.96 F1 versus Gandalf's 0.92 at 23% lower cost, with a wider margin on process criteria."


## Visualizando

In [20]:
# pega o abstract com mais entidades de domínio
melhor = max(docs, key=lambda d: sum(1 for ent in d.ents if ent.label_ in ROTULOS))

cores = {"MODEL": "#a5d8ff", "DATASET": "#b2f2bb", "METRIC": "#ffd8a8"}
displacy.render(melhor, style="ent", options={"ents": ROTULOS, "colors": cores})

## Salvando o pipeline e as entidades

In [21]:
# salva o pipeline completo. Pra carregar depois (spacy.load("data/ner_arxiv")) a função modelo_regex precisa
# estar registrada na sessão, ou seja, a célula do @Language.component tem que ter rodado antes (igual na live 8)
nlp.to_disk("data/ner_arxiv")

In [22]:
# exporta as entidades de domínio de cada paper, serve de entrada pra uma próxima etapa (ex: RAG)
saida = []
for paper_id, doc in zip(ids, docs):
    saida.append({
        "id": paper_id,
        "entidades": [{"texto": ent.text, "label": ent.label_} for ent in doc.ents if ent.label_ in ROTULOS],
    })

with open("data/entidades_arxiv.json", "w", encoding="utf-8") as f:
    json.dump(saida, f, ensure_ascii=False, indent=2)

print(len(saida), "papers ->", "data/entidades_arxiv.json")

340 papers -> data/entidades_arxiv.json


## Papers parecidos com similaridade

Mesma ideia da aula 3: o `en_core_web_md` tem vetores de palavras, então `doc.similarity()` funciona de verdade.

In [23]:
# o modelo médio inclui os vetores (o sm não tem)
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 5.2 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')


In [24]:
nlp_md = spacy.load("en_core_web_md")
docs_md = list(nlp_md.pipe(textos[:100]))

consulta = 0  # índice do abstract de referência (troca à vontade)
print("Referência:", titulos[consulta])

# ordena todos pela similaridade com o de referência
ranking = sorted(range(len(docs_md)), key=lambda i: docs_md[consulta].similarity(docs_md[i]), reverse=True)

# o [0] é o próprio abstract, por isso começa do 1
for i in ranking[1:4]:
    print(round(docs_md[consulta].similarity(docs_md[i]), 3), titulos[i])

Referência: Agent Behavior as Code: Efficient and Robust LLM Agents with Programmatic Specifications
0.975 Greedy Local Learning for Language Model Pretraining: Gaps and Objective Design
0.971 A Graph-Based Inspection and Intervention Tool for Assessing Mechanistic Learning in PINNs
0.971 Viva La Vida: Verification and Accumulation Failures in Multi-Agent Proof Search
